# Post-Fire Debris Flow Hazard Pipeline
### 00 — Model driver and sanity checks

This notebook is a **driver**, not the project. The real logic lives in
`src/debrisflow/`, where it is version controlled and unit tested. The
notebook's only jobs are to pull that code, run it, and show results.

**Why the split:** notebooks re-run cells out of order, which silently changes
results. Fine for looking at a raster. Not fine for a hazard model. So the math
lives in `.py` files with tests, and this notebook just calls it.

**What this project is:** a Sentinel-2 to web-map pipeline built around the
USGS operational post-fire debris-flow models, cross-validated against the
reference implementation, with a sensitivity analysis USGS does not publish.
The value is in the ingest and the delivery, not in the equations.

**Modules so far**

| Module | Produces | Status |
|---|---|---|
| `m1.py` | debris-flow likelihood from T, F, S, R | tested |
| `severity.py` | dNBR and the **F** variable, from Sentinel-2 | tested |
| `terrain.py` | slope, basins, and the **T** variable, from a DEM | tested |
| `soils.py` | the **S** variable, from SSURGO/STATSGO | not started |

## 1. Pull the code from GitHub

Colab gives you a fresh machine whenever the runtime restarts and nothing
persists. That sounds annoying but it is a useful guarantee: if this notebook
runs clean top to bottom, everything it needs is genuinely committed. Anything
you forgot to push simply vanishes.

**Edit `GITHUB_USER` below.**

In [1]:
GITHUB_USER = "jerrod-lessel"   # <-- change this if you're not me :-)
REPO = "postfire-debris-flow"

# Always start from /content, never from wherever a previous run left us.
# %cd is persistent, so without this line re-running the cell clones a repo
# inside the repo and everything downstream loads stale code.
%cd /content

!rm -rf {REPO}
!git clone -q https://github.com/{GITHUB_USER}/{REPO}.git
%cd /content/{REPO}

!pwd                    # should be exactly /content/postfire-debris-flow
!git log -1 --oneline   # should show your latest commit

/content
/content/postfire-debris-flow
/content/postfire-debris-flow
4a28b2a (HEAD -> main, origin/main, origin/HEAD) Update README.md


## 2. Install dependencies

No runtime restart needed, and no numpy pin. 🎉

**The problem we are working around:** pysheds 0.5 calls `np.in1d`, which numpy
2.0 removed in favour of the identical `np.isin`. On a default Colab runtime
(numpy 2.x) `grid.accumulation()` therefore dies with
`AttributeError: module 'numpy' has no attribute 'in1d'`.

**Why we do not pin numpy<2:** rasterio, geopandas and opencv all now require
numpy>=2, so downgrading forces a slow source build and breaks other packages.
Instead `debrisflow/_compat.py` restores the single removed alias. It is a true
alias, not an approximation, and there is a regression test that runs real D8
flow accumulation to prove it works.

In [2]:
!pip install -q -r requirements.txt

import numpy as np
print("numpy", np.__version__)

# The shim is applied automatically when debrisflow.terrain is imported.
import sys; sys.path.insert(0, "src")
from debrisflow._compat import PATCHED
print("np.in1d shim applied:", PATCHED, "(False just means numpy already had it)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.6 MB/s eta 0:00:00
numpy 2.1.3
np.in1d shim applied: False (False just means numpy already had it)


### Already cloned? Use this instead

Once the repo is on the machine you do not need to re-clone after every edit.
Commit on GitHub, then pull just the new commits.

In [4]:
!git pull -q
!git log -1 --oneline    # confirm you actually got your change

56e3b65 (HEAD -> main, origin/main, origin/HEAD) Update test_basins.py


## 3. Run the test suite 🧪

The most important cell here. The tests pin down that probability rises with
rainfall, that slope is computed correctly against known planes, that the
terrain variable is an intersection rather than a product, that the Sentinel-2
baseline offset is applied, and that bad pixels fail loudly rather than
quietly.

Run this **before** trusting any map you make. A hazard map built on broken
math looks exactly like one built on correct math, which is what makes it
dangerous.

Expected: `151 passed`.

In [5]:
!python -m pytest -q

........................................................................ [ 47%]
..s..................................................................... [ 94%]
........                                                                 [100%]
151 passed, 1 skipped in 6.15s


## 4. Import the modules

`sys.path.insert(0, "src")` tells Python to look inside `src/` when resolving
imports, which is what makes `from debrisflow...` work here. (Pytest already
knows this from `pyproject.toml`; the notebook does not.)

In [ ]:
import sys

# Absolute path, not "src". A relative path resolves against whatever the
# working directory happens to be at import time, which is exactly the hidden
# state that causes nested-clone problems. Absolute cannot drift.
sys.path.insert(0, "/content/postfire-debris-flow/src")

import numpy as np

from debrisflow.m1 import (
    BasinVariables,        # holds the three predictors T, F, S for one basin
    RainfallConvention,    # accumulation (mm) vs intensity (mm/hr)
    likelihood,            # forward model  -> probability
    accumulation,          # inverted model -> rainfall needed for a given p
    hazard_class,          # probability    -> Low / Moderate / High
)
from debrisflow.severity import (
    nbr, dnbr, to_reflectance, needs_baseline_offset,
    moderate_high_mask, basin_F,
)
from debrisflow.terrain import (
    slope_degrees, steep_mask, basin_T, basin_area_km2, in_calibration_range,
)

# Shorthand for the two rainfall conventions.
MM_HR = RainfallConvention.INTENSITY_MM_HR   # peak intensity, mm per hour
MM    = RainfallConvention.ACCUMULATION_MM   # accumulation, mm over duration

# Tripwire: confirms which copy of the package actually loaded.
import debrisflow
print("loaded from:", debrisflow.__file__)
print("numpy:", np.__version__)
print("modules imported.")

loaded from: /content/postfire-debris-flow/src/debrisflow/__init__.py
numpy: 2.1.3
modules imported.


## 5. The three predictors

Everything in this project exists to compute these three numbers per basin.
The M1 coefficients were fitted to these **exact** definitions, so getting the
definitions right matters more than anything else in the pipeline.

| Var | Definition | Source module | Typical range |
|---|---|---|---|
| **T** | Fraction of basin that is **both** ≥23° slope **and** burned at moderate/high severity | `terrain.py` | 0.0 – 0.8 |
| **F** | Mean **dNBR ÷ 1000** over the basin | `severity.py` | 0.1 – 0.8 |
| **S** | Area-weighted mean soil **KF-factor** | `soils.py` (todo) | 0.02 – 0.55 |

⚠️ **The trap in T:** it is an *intersection*, not two separate proportions.
A basin can be 90% steep and 90% burned while the steep parts and the burned
parts barely overlap. The model cares where they coincide, because that is
where rilling and dry ravel deliver sediment to the channel. Section 7
quantifies how badly this goes wrong.

## 6. Rainfall units: resolved ✅

M1's rainfall term **R is an accumulation in millimetres** over the duration,
not an intensity in mm/hr. Confirmed from the USGS `pfdf` documentation for
`models.staley2017`, which gives the model as
X = B + Ct·T·R + Cf·F·R + Cs·S·R with *"R: Rainfall accumulation in mm"*.

Over 15 minutes the two conventions differ by a factor of **4**. Feed the wrong
one in and the model still returns a number between 0 and 1 and still draws a
convincing map.

🔴 **The trap gets worse later.** The USGS docs also note that `staley2017`
works strictly with accumulations while `gartner2014` (sediment volume) expects
*intensities*. If we add volume modelling, this pipeline will carry both
conventions at once. That is why `likelihood()` has **no default** for its
convention argument: you must state your units at every call site.

In [ ]:
# A middling basin: not obviously dangerous, not obviously safe.
b = BasinVariables(T=0.15, F=0.35, S=0.25)
rain = 12   # the same number, read two different ways

p_intensity = likelihood(b, rain, MM_HR)  # correct: 12 mm/hr -> 3 mm accumulated
p_accum     = likelihood(b, rain, MM)     # wrong: treated as 12 mm accumulated

print(f"as intensity (correct) : p = {p_intensity:.3f}  -> {hazard_class(p_intensity)}")
print(f"as accumulation (wrong): p = {p_accum:.3f}  -> {hazard_class(p_accum)}")
print("\nSame basin, same rain figure, opposite operational decision.")

as intensity (correct) : p = 0.098  -> Low
as accumulation (wrong): p = 0.883  -> High

Same basin, same rain figure, opposite operational decision.


## 7. Why T must be an intersection

Here we build a synthetic burn scar where the severe burn and the steep terrain
partially overlap, as they do in reality, and compute T both ways.

In [ ]:
# Synthetic 200x200 scene at 10 m: a severity blob and a steepness blob,
# offset from each other so they only partly overlap.
rng = np.random.default_rng(7)
yy, xx = np.mgrid[0:200, 0:200]

dnbr_vals = 900 * np.exp(-(((xx-100)**2 + (yy-100)**2) / (2*55**2)))
dnbr_vals += rng.normal(0, 60, (200, 200))

slope_vals = 35 * np.exp(-(((xx-120)**2 + (yy-90)**2) / (2*70**2)))
slope_vals += rng.normal(0, 4, (200, 200))

basin = np.ones((200, 200), dtype=bool)

burned = moderate_high_mask(dnbr_vals)   # dNBR >= 270
steep  = steep_mask(slope_vals)          # slope >= 23 degrees

T_correct = basin_T(steep, burned, basin)          # the intersection
T_naive   = steep.mean() * burned.mean()           # the original spec's error

print(f"steep fraction        : {steep.mean():.3f}")
print(f"burned fraction       : {burned.mean():.3f}")
print(f"T, intersection (right): {T_correct:.3f}")
print(f"T, product      (wrong): {T_naive:.3f}")
print(f"\nUnderestimate: {(1 - T_naive/T_correct)*100:.0f}%")

steep fraction        : 0.334
burned fraction       : 0.580
T, intersection (right): 0.311
T, product      (wrong): 0.194

Underestimate: 38%


In [ ]:
# What that error does to the actual hazard call.
F = basin_F(dnbr_vals, basin)
S = 0.28   # placeholder until soils.py exists

print(f"F = {F:.3f}   S = {S:.3f} (placeholder)\n")
print(f"{'I15 (mm/hr)':>12} {'p (correct T)':>16} {'p (wrong T)':>14}")
print("-" * 46)
for i in (12, 24, 40):
    pc = likelihood(BasinVariables(T_correct, F, S), i, MM_HR)
    pw = likelihood(BasinVariables(T_naive,   F, S), i, MM_HR)
    print(f"{i:>12} {pc:>10.3f} ({hazard_class(pc)[0]}) {pw:>9.3f} ({hazard_class(pw)[0]})")

v = BasinVariables(T_correct, F, S)
print(f"\nI15 needed for p=0.5: {accumulation(v, 0.5):.1f} mm/hr")

F = 0.370   S = 0.280 (placeholder)

 I15 (mm/hr)    p (correct T)    p (wrong T)
----------------------------------------------
          12      0.128 (L)     0.113 (L)
          24      0.450 (M)     0.380 (M)
          40      0.890 (H)     0.833 (H)

I15 needed for p=0.5: 25.4 mm/hr


## 8. Slope, checked against known geometry

Slope uses Horn's method, the same 3×3 kernel GDAL and ArcGIS use, so results
are comparable to standard GIS output. The tests verify it against planes whose
slope we know analytically, which is stronger than eyeballing a hillshade.

Note the resolution caveat: the USGS models were calibrated on **10 m** DEM
data, and slope statistics are resolution dependent. A coarser DEM smooths away
exactly the steep pixels M1 cares about, so 3DEP 10 m is the right input.

In [ ]:
# A plane rising 10 m per 10 m cell is exactly 45 degrees, by definition.
plane = np.tile(np.arange(6) * 10.0, (6, 1))
print(f"45-degree plane -> {np.nanmax(slope_degrees(plane, 10)):.4f} degrees")

# Same elevations, coarser cells: the same terrain reads as gentler.
for cs in (10, 30, 90):
    print(f"  cellsize {cs:>2} m -> max slope {np.nanmax(slope_degrees(plane, cs)):.1f} deg")

# On our synthetic scene, how much is steep enough for M1 to care?
print(f"\nsynthetic scene: {steep.mean()*100:.0f}% of pixels are >= 23 degrees")

45-degree plane -> 45.0000 degrees
  cellsize 10 m -> max slope 45.0 deg
  cellsize 30 m -> max slope 18.4 deg
  cellsize 90 m -> max slope 6.3 deg

synthetic scene: 33% of pixels are >= 23 degrees


## 9. Basin scale, the thing most likely to break agreement with USGS

M1 was calibrated on small watersheds, roughly **0.1 to 8 km²**. Larger basins
average burn severity and steepness over terrain that never contributes
sediment to the channel, which dilutes T toward zero and systematically
**under-predicts** hazard.

This is the main argument for delineating our own basins with pysheds rather
than using off-the-shelf NHDPlus HR catchments, which are often far larger.
Basins outside the calibration range still produce a number, but that number is
an extrapolation, so we flag them rather than dropping them silently.

In [ ]:
# What the calibration range means in pixels, and how basins get flagged.
for area in (0.05, 0.5, 2.0, 8.0, 45.0):
    n_cells = int(area * 1e6 / 100)   # 10 m cells
    ok = in_calibration_range(area)
    note = "in range" if ok else "EXTRAPOLATION - flag on the map"
    print(f"{area:>6.2f} km2 = {n_cells:>7} cells   {note}")

  0.05 km2 =     500 cells   EXTRAPOLATION - flag on the map
  0.50 km2 =    5000 cells   in range
  2.00 km2 =   20000 cells   in range
  8.00 km2 =   80000 cells   in range
 45.00 km2 =  450000 cells   EXTRAPOLATION - flag on the map


## 10. Verify Soil Data Access 🌍

Everything above runs on synthetic arrays. This is the first cell that touches
a live external service.

**Why this cell exists separately:** the KF-factor aggregation math in
`soils.py` is unit tested, but the Soil Data Access SQL and endpoint are not.
They were written from documentation, not verified against the running
service. If the schema has drifted or the service is down, we want to know here
with a readable error, rather than three steps later when every basin quietly
comes back with `S = NaN`.

Soil Data Access is a USDA REST endpoint that takes a SQL query and returns
tabular soil data. We ask it for a single row from the STATSGO legend, which is
the cheapest possible proof that the connection works and the area symbol is
still `US`.

**Expected:** a single legend row. **If it fails,** check
https://sdmdataaccess.sc.egov.usda.gov/ before doing any soil work.

In [ ]:
from debrisflow.soils import check_sda_connection, sda_query, STATSGO_AREASYMBOL

try:
    rows = check_sda_connection()
    print("SDA reachable.")
    print(f"  STATSGO area symbol {STATSGO_AREASYMBOL!r} ->", rows[0])
except Exception as e:
    print("SDA CHECK FAILED:", type(e).__name__, e)
    print("\nDo not trust any S values until this passes.")

SDA reachable.
  STATSGO area symbol 'US' -> ['US', 'United States']


### If that worked, try a real KF lookup

A second, slightly harder check: pull actual surface-horizon KF values for a
handful of STATSGO map units. This exercises the real query builder rather than
a canned smoke-test string, so it catches schema problems the first cell would
miss.

In [ ]:
# Grab a few STATSGO map unit keys, then fetch their soil horizons.
try:
    mu_rows = sda_query(
        "SELECT TOP 3 mu.mukey FROM legend l "
        "INNER JOIN mapunit mu ON l.lkey = mu.lkey "
        f"WHERE l.areasymbol = '{STATSGO_AREASYMBOL}'"
    )
    mukeys = [r[0] for r in mu_rows]
    print("map unit keys:", mukeys)

    from debrisflow.soils import kf_by_mapunit_sql
    rows = sda_query(kf_by_mapunit_sql(mukeys, dataset="statsgo"))

    print(f"\n{len(rows)} horizon rows returned")
    print(f"{'mukey':>10} {'comppct':>9} {'kffact':>8} {'top_cm':>8}")
    print("-" * 40)
    for r in rows[:8]:
        mukey, cokey, comppct, kf, top, bot = r
        print(f"{mukey:>10} {str(comppct):>9} {str(kf):>8} {str(top):>8}")
    print("\nNulls in kffact are normal (rock outcrop, water, organic soils).")
    print("soils.py drops them and renormalises rather than treating them as 0.")
except Exception as e:
    print("KF LOOKUP FAILED:", type(e).__name__, e)

map unit keys: ['662058', '666638', '666643']

94 horizon rows returned
     mukey   comppct   kffact   top_cm
----------------------------------------
    662058        54      .37        0
    662058        54      .37       10
    662058        54      .28       38
    662058        54      .10       64
    662058        18      .37        0
    662058        18      .24       43
    662058        18      .24       69
    662058        15      .32        0

Nulls in kffact are normal (rock outcrop, water, organic soils).
soils.py drops them and renormalises rather than treating them as 0.


## What's next

1. **`soils.py`** — the S variable. USGS operationally uses STATSGO KF-factor;
   SSURGO is finer but a deviation from the calibration data, so whichever we
   choose gets stated openly in the writeup.
2. **Real data** — STAC query for pre/post Sentinel-2 over the 2024 Bridge Fire,
   3DEP DEM, and pysheds delineation on actual terrain.
3. **Cross-validation** — install the official USGS `pfdf` and assert our
   numbers match theirs. That test is the strongest validation artifact in the
   project.
4. **Sensitivity analysis** — which basins are High under *every* assumption,
   and which flip depending on dNBR threshold and basin delineation. This is
   the part USGS does not publish.
5. **Delivery** — tippecanoe to PMTiles, COGs, Cloudflare R2, MapLibre.